In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import numpy as np
import random
import os
import logging
from datetime import datetime

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

CONFIG = {
    'batch_size': 32,
    'embedding_dim': 128,
    'hidden_dim': 256,
    'num_layers': 2,
    'dropout': 0.3,
    'learning_rate': 0.001,
    'epochs': 20,
    'max_seq_len': 15,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'min_freq': 2,
}

print(f"Using device: {CONFIG['device']}")
print(f"Batch size: {CONFIG['batch_size']}")
print(f"Embedding dim: {CONFIG['embedding_dim']}")
print(f"Hidden dim: {CONFIG['hidden_dim']}")

Using device: cpu
Batch size: 32
Embedding dim: 128
Hidden dim: 256


In [2]:
#Load and preprocess parallel sentences
from google.colab import drive
drive.mount('/content/drive')

#For checkpoint, using small dataset from manythings.org
!wget http://www.manythings.org/anki/fra-eng.zip
!unzip -q fra-eng.zip

def load_sentences(file_path, max_pairs=15000):
    english = []
    french = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= max_pairs:
                break
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                eng = parts[0].lower().strip()
                fra = parts[1].lower().strip()
                english.append(eng)
                french.append(fra)
    return english, french

eng_sentences, fra_sentences = load_sentences('fra.txt', max_pairs=15000)
print(f"Loaded {len(eng_sentences)} sentence pairs")
print(f"Example - English: {eng_sentences[0]}")
print(f"Example - French: {fra_sentences[0]}")

Mounted at /content/drive
--2026-04-19 05:09:09--  http://www.manythings.org/anki/fra-eng.zip
Resolving www.manythings.org (www.manythings.org)... 173.254.30.110
Connecting to www.manythings.org (www.manythings.org)|173.254.30.110|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8242175 (7.9M) [application/zip]
Saving to: ‘fra-eng.zip’

fra-eng.zip         100%[===================>]   7.86M  21.0MB/s    in 0.4s    

2026-04-19 05:09:09 (21.0 MB/s) - ‘fra-eng.zip’ saved [8242175/8242175]

Loaded 15000 sentence pairs
Example - English: go.
Example - French: va !


In [3]:
#Build vocabulary from sentences
def build_vocab(sentences, min_freq=1, max_size=10000):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.split())

    #Filter by min_freq
    if min_freq > 1:
        filtered = {word: count for word, count in counter.items() if count >= min_freq}
    else:
        filtered = counter

    #Take most common up to max_size
    most_common = sorted(filtered.items(), key=lambda x: x[1], reverse=True)[:max_size]

    word2idx = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}
    idx2word = {0: '<PAD>', 1: '<UNK>', 2: '<SOS>', 3: '<EOS>'}

    for idx, (word, _) in enumerate(most_common, start=4):
        word2idx[word] = idx
        idx2word[idx] = word

    return word2idx, idx2word

eng_word2idx, eng_idx2word = build_vocab(eng_sentences)
fra_word2idx, fra_idx2word = build_vocab(fra_sentences)

print(f"English vocab size: {len(eng_word2idx)}")
print(f"French vocab size: {len(fra_word2idx)}")
print(f"Sample English words: {list(eng_word2idx.items())[:10]}")

English vocab size: 3725
French vocab size: 7264
Sample English words: [('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('i', 4), ('tom', 5), ("i'm", 6), ('you', 7), ('a', 8), ('is', 9)]


In [4]:
#Convert sentences to tensor indices
def encode_sentence(sentence, word2idx, max_len):
    tokens = sentence.split()
    indices = [word2idx.get(token, word2idx['<UNK>']) for token in tokens]
    #Truncate if too long
    if len(indices) > max_len - 1:
        indices = indices[:max_len - 1]
    #Add <EOS> token
    indices.append(word2idx['<EOS>'])
    #Pad
    indices = indices + [word2idx['<PAD>']] * (max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, eng_sentences, fra_sentences, eng_word2idx, fra_word2idx, max_len):
        self.eng_sentences = eng_sentences
        self.fra_sentences = fra_sentences
        self.eng_word2idx = eng_word2idx
        self.fra_word2idx = fra_word2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.eng_sentences)

    def __getitem__(self, idx):
        eng = encode_sentence(self.eng_sentences[idx], self.eng_word2idx, self.max_len)
        fra = encode_sentence(self.fra_sentences[idx], self.fra_word2idx, self.max_len)
        return eng, fra

#Split into train/val
split_idx = int(0.9 * len(eng_sentences))
train_eng, train_fra = eng_sentences[:split_idx], fra_sentences[:split_idx]
val_eng, val_fra = eng_sentences[split_idx:], fra_sentences[split_idx:]

train_dataset = TranslationDataset(train_eng, train_fra, eng_word2idx, fra_word2idx, CONFIG['max_seq_len'])
val_dataset = TranslationDataset(val_eng, val_fra, eng_word2idx, fra_word2idx, CONFIG['max_seq_len'])

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Max sequence length: {CONFIG['max_seq_len']}")

Train samples: 13500
Val samples: 1500
Max sequence length: 15


In [18]:
#Encoder with LSTM (bidirectional)
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, (hidden, cell) = self.lstm(embedded)
        #hidden shape: (2*num_layers, batch, hidden_dim)
        #Reshape to (num_layers, batch, 2*hidden_dim) for decoder
        hidden = hidden.view(self.num_layers, 2, -1, self.hidden_dim)
        hidden = hidden.permute(1, 0, 2, 3).contiguous()
        hidden = hidden.view(2, -1, self.hidden_dim * 2)
        cell = cell.view(self.num_layers, 2, -1, self.hidden_dim)
        cell = cell.permute(1, 0, 2, 3).contiguous()
        cell = cell.view(2, -1, self.hidden_dim * 2)
        return outputs, hidden, cell

#Decoder with attention
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim * 2, num_layers,
                            batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden_dim * 4, 1)
        self.fc_out = nn.Linear(hidden_dim * 4, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.hidden_dim = hidden_dim

    def forward(self, x, hidden, cell, encoder_outputs):
        x = x.unsqueeze(1)
        embedded = self.dropout(self.embedding(x))
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        #Attention mechanism
        seq_len = encoder_outputs.shape[1]
        hidden_repeated = hidden[-1].unsqueeze(1).repeat(1, seq_len, 1)
        concat = torch.cat((hidden_repeated, encoder_outputs), dim=2)
        attention_weights = torch.softmax(self.attention(concat), dim=1)
        context = torch.sum(attention_weights * encoder_outputs, dim=1).unsqueeze(1)

        concat_final = torch.cat((output, context), dim=2)
        prediction = self.fc_out(concat_final)
        return prediction.squeeze(1), hidden, cell

#Complete Seq2Seq model
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        trg_len = trg.shape[1]
        batch_size = trg.shape[0]
        trg_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)
        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1

        return outputs

#Reinitialize model
encoder = Encoder(len(eng_word2idx), CONFIG['embedding_dim'], CONFIG['hidden_dim'],
                  CONFIG['num_layers'], CONFIG['dropout'])
decoder = Decoder(len(fra_word2idx), CONFIG['embedding_dim'], CONFIG['hidden_dim'],
                  CONFIG['num_layers'], CONFIG['dropout'])
model = Seq2Seq(encoder, decoder, CONFIG['device']).to(CONFIG['device'])

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model has {count_parameters(model):,} trainable parameters")
print(f"Encoder: {count_parameters(encoder):,}")
print(f"Decoder: {count_parameters(decoder):,}")

Model has 14,636,769 trainable parameters
Encoder: 2,844,288
Decoder: 11,792,481


In [19]:
#Initialize optimizer, loss function, and scheduler
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
criterion = nn.CrossEntropyLoss(ignore_index=0)

#For tracking metrics
train_losses = []
val_losses = []

def train_epoch(model, dataloader, optimizer, criterion, teacher_forcing_ratio):
    model.train()
    total_loss = 0
    for batch_idx, (src, trg) in enumerate(dataloader):
        src = src.to(CONFIG['device'])
        trg = trg.to(CONFIG['device'])

        optimizer.zero_grad()
        output = model(src, trg, teacher_forcing_ratio)

        #Reshape for loss calculation
        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        #Gradient clipping to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, trg in dataloader:
            src = src.to(CONFIG['device'])
            trg = trg.to(CONFIG['device'])
            output = model(src, trg, teacher_forcing_ratio=0)

            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            trg = trg[:, 1:].reshape(-1)

            loss = criterion(output, trg)
            total_loss += loss.item()

    return total_loss / len(dataloader)

print("Training setup complete")
print(f"Optimizer: Adam (lr={CONFIG['learning_rate']})")
print(f"Scheduler: ReduceLROnPlateau (factor=0.5, patience=2)")
print(f"Loss: CrossEntropyLoss (ignore_index=0 for PAD)")
print(f"Gradient clipping: max_norm=1")

Training setup complete
Optimizer: Adam (lr=0.001)
Scheduler: ReduceLROnPlateau (factor=0.5, patience=2)
Loss: CrossEntropyLoss (ignore_index=0 for PAD)
Gradient clipping: max_norm=1


In [21]:
#Training loop with few epochs
best_val_loss = float('inf')
patience_counter = 0
early_stop_patience = 3

print("Starting training")

for epoch in range(5):
    teacher_ratio = 0.5  #Fixed teacher forcing

    train_loss = train_epoch(model, train_loader, optimizer, criterion, teacher_ratio)
    val_loss = evaluate(model, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1}/5 | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}")

    #Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_nmt_model.pt')
        print(f"Saved best model (val_loss: {val_loss:.4f})")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= early_stop_patience:
        print(f"Early stopping triggered")
        break

print(f"Training complete. Best validation loss: {best_val_loss:.4f}")

Starting training
Epoch 1/5 | Train Loss: 2.5321 | Val Loss: 4.5903 | LR: 0.001000
Saved best model (val_loss: 4.5903)
Epoch 2/5 | Train Loss: 1.9152 | Val Loss: 4.6685 | LR: 0.001000
Epoch 3/5 | Train Loss: 1.5133 | Val Loss: 4.6109 | LR: 0.001000
Epoch 4/5 | Train Loss: 1.2629 | Val Loss: 4.6530 | LR: 0.000500
Early stopping triggered
Training complete. Best validation loss: 4.5903


In [23]:
#Translation function for inference
def translate_sentence(model, sentence, eng_word2idx, fra_idx2word, max_len=15, device='cpu'):
    model.eval()

    #Tokenize and encode source sentence
    tokens = sentence.lower().split()
    indices = [eng_word2idx.get(token, eng_word2idx['<UNK>']) for token in tokens]
    indices = indices[:max_len-1]
    indices.append(eng_word2idx['<EOS>'])
    indices = indices + [eng_word2idx['<PAD>']] * (max_len - len(indices))

    src_tensor = torch.tensor(indices).unsqueeze(0).to(device)

    #Encode
    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src_tensor)

        #Start with <SOS> token
        trg_index = [fra_word2idx['<SOS>']]

        for _ in range(max_len):
            trg_tensor = torch.tensor([trg_index[-1]]).to(device)
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell, encoder_outputs)
            pred_token = output.argmax(1).item()
            trg_index.append(pred_token)

            if pred_token == fra_word2idx['<EOS>']:
                break

    #Convert indices to words
    translation = []
    for idx in trg_index[1:-1]:  #Skip <SOS> and <EOS>
        if idx == fra_word2idx['<PAD>']:
            break
        translation.append(fra_idx2word.get(idx, '<UNK>'))

    return ' '.join(translation)

#Test on some examples
test_sentences = [
    "hello",
    "how are you",
    "i love you",
    "good morning",
    "what is your name"
]

print("Sample translations:")
for sent in test_sentences:
    translation = translate_sentence(model, sent, eng_word2idx, fra_idx2word,
                                      CONFIG['max_seq_len'], CONFIG['device'])
    print(f"EN: {sent}")
    print(f"FR: {translation}")
    print()

Sample translations:
EN: hello
FR: !

EN: how are you
FR: seule ?

EN: i love you
FR: lundi

EN: good morning
FR: avec une

EN: what is your name
FR: est-il ?



In [25]:
#BLEU score evaluation
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

def compute_bleu(model, dataloader, eng_word2idx, fra_idx2word, fra_word2idx, device):
    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch_idx, (src, trg) in enumerate(dataloader):
            src = src.to(device)
            trg = trg.to(device)

            for i in range(src.shape[0]):
                src_sent = src[i]

                encoder_outputs, hidden, cell = model.encoder(src_sent.unsqueeze(0))

                trg_idx = [fra_word2idx['<SOS>']]
                for _ in range(CONFIG['max_seq_len']):
                    trg_tensor = torch.tensor([trg_idx[-1]]).to(device)
                    output, hidden, cell = model.decoder(trg_tensor, hidden, cell, encoder_outputs)
                    pred = output.argmax(1).item()
                    trg_idx.append(pred)
                    if pred == fra_word2idx['<EOS>']:
                        break

                ref = []
                for idx in trg[i].tolist():
                    if idx == fra_word2idx['<EOS>']:
                        break
                    if idx != fra_word2idx['<SOS>'] and idx != fra_word2idx['<PAD>']:
                        ref.append(fra_idx2word.get(idx, '<UNK>'))

                hyp = []
                for idx in trg_idx[1:-1]:
                    if idx == fra_word2idx['<PAD>']:
                        break
                    hyp.append(fra_idx2word.get(idx, '<UNK>'))

                if len(hyp) > 0 and len(ref) > 0:
                    references.append(ref)
                    hypotheses.append(hyp)

            if batch_idx >= 50:
                break

    smoothie = SmoothingFunction().method4
    bleu_score = corpus_bleu([[ref] for ref in references], hypotheses, smoothing_function=smoothie)
    return bleu_score, references[:5], hypotheses[:5]

print("Computing BLEU score on validation set")
bleu, ref_samples, hyp_samples = compute_bleu(model, val_loader, eng_word2idx, fra_idx2word, fra_word2idx, CONFIG['device'])

print(f"\nBLEU-4 Score: {bleu:.4f}")
print("\nSample predictions vs references:")
for i in range(min(3, len(ref_samples))):
    print(f"Reference: {' '.join(ref_samples[i])}")
    print(f"Predicted: {' '.join(hyp_samples[i])}")
    print()

Computing BLEU score on validation set

BLEU-4 Score: 0.0123

Sample predictions vs references:
Reference: tom sourit.
Predicted: en haut.

Reference: tom sanglote.
Predicted: en train de

Reference: tom est en sanglots.
Predicted: en train de

